In [1]:
from elasticsearch import Elasticsearch
import json
import requests

In [2]:
ES_HOST = "https://prj-ext-prod-trec-dp-452111.es.europe-west2.gcp.elastic-cloud.com"
ES_USERNAME = "elastic"
ES_PASSWORD = "qhCaiXhgjBIDR939OLU6beoz"

In [3]:
biosamples_root_url = "https://www.ebi.ac.uk/biosamples/samples"

In [4]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))

In [5]:
samples = {}

In [6]:
project_tag = "Traversing%20European%20Coastlines%20(TREC)%20expedition"
first_url = (
            f"{biosamples_root_url}?size=200&filter="
            f"attr%3Aproject%3A{project_tag}"
        )
samples_response = requests.get(first_url, timeout=60).json()

In [9]:
first_url

'https://www.ebi.ac.uk/biosamples/samples?size=200&filter=attr%3Aproject%3ATraversing%20European%20Coastlines%20(TREC)%20expedition'

In [7]:
while "_embedded" in samples_response:
    for sample in samples_response["_embedded"]["samples"]:
        samples[sample["accession"]] = sample
    if "next" in samples_response["_links"]:
        samples_response = requests.get(samples_response["_links"]["next"]["href"], timeout=60).json()
    else:
        samples_response = requests.get(samples_response["_links"]["last"]["href"], timeout=60).json()

In [8]:
len(samples)

178

In [10]:
from collections import defaultdict

In [12]:
samples[0]

KeyError: 0

In [14]:
dates = defaultdict(int)
for biosample_id, sample in samples.items():
    dates[sample["update"]] += 1

In [21]:
sorted(list(dates.keys()))[-10:]

['2025-03-13T11:58:47.730Z',
 '2025-03-13T11:58:47.749Z',
 '2025-03-13T11:58:47.768Z',
 '2025-03-13T11:58:47.782Z',
 '2025-03-13T11:58:47.791Z',
 '2025-03-13T11:58:47.799Z',
 '2025-03-13T11:58:47.817Z',
 '2025-03-13T11:58:47.835Z',
 '2025-03-13T11:58:47.851Z',
 '2025-03-13T11:58:47.878Z']

In [22]:
for biosample_id, sample in samples.items():
    if sample["update"] == "2025-03-13T11:58:47.878Z":
        print(biosample_id)

SAMEA112561342


In [73]:
def check_field_existence(record):
    values = list()
    units = list()
    ontology_terms = list()
    for element in record:
        values.append(element["text"])
        try:
            units.append(element["unit"])
        except KeyError:
            pass
        try:
            ontology_terms.append(element["ontologyTerms"][0])
        except KeyError:
            pass
    return ", ".join(values), ", ".join(units), ", ".join(ontology_terms)

In [74]:
mandatory_fields = ["organism", "depth", "collection date", "altitude", "geographic location (latitude)", "geographic location (longitude)", 
                    "geographic location (country and/or sea)"]

In [75]:
samples["SAMEA112488125"]["relationships"]

[{'source': 'SAMEA112488125',
  'type': 'derived from',
  'target': 'SAMEA117117492'}]

In [76]:
actions = []
columns_mapping = {"collection date": "collection_date", "geographic location (latitude)": "lat", "geographic location (longitude)": "lon", "geographic location (country and/or sea)": "location"}
for sample_id, sample in samples.items():
    item = dict()
    item["customFields"] = []
    for record_name, record in sample["characteristics"].items():
        values, units, _ = check_field_existence(record)
        if record_name not in mandatory_fields:
            item["customFields"].append(
                {
                    "name": record_name,
                    "value": values,
                    "unit": units
                }
            )
        else:
            if record_name == "collection date":
                try:
                    values = parser.parse(values)
                except parser.ParserError:
                    values = None
            if record_name in ["geographic location (latitude)", "geographic location (longitude)"]:
                try:
                    values = float(values)
                except ValueError:
                    values = None
            if units != '':
                values = f"{values} {units}"
            if record_name in columns_mapping:
                item[columns_mapping[record_name]] = values
            else:
                item[record_name] = values
    if "relationships" in sample:
        item["relationships"] = sample["relationships"]
    else:
        item["relationships"] = []
    item["biosampleId"] = sample_id
    actions.append({"index": {"_index": "2025-03-24_data_portal", "_id": sample_id}})
    actions.append(item)

In [79]:
for i in range(0, len(actions), 10000):
    print(f"Working on {i}:{i+10000}")
    _ = es.bulk(body=actions[i:i+10000])

Working on 0:10000
Working on 10000:20000
Working on 20000:30000
Working on 30000:40000
Working on 40000:50000
Working on 50000:60000
Working on 60000:70000
Working on 70000:80000
Working on 80000:90000
Working on 90000:100000
Working on 100000:110000
Working on 110000:120000
